# Data Overview EDA

This notebook does a first-pass review of the social media users dataset.

Focus areas:

- load the raw dataset
- check missing values and duplicate users
- parse and inspect interests
- review DOB, gender, and location fields
- capture preprocessing decisions for the recommendation system


In [1]:
from pathlib import Path
import ast
import pandas as pd

DATA_PATH_CANDIDATES = [
    Path("../data/SocialMediaUsersDataset.csv"),   # when notebook is run from EDA/
    Path("data/SocialMediaUsersDataset.csv"),      # when notebook is run from repo root
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find SocialMediaUsersDataset.csv in data/ or ../data/")

DATA_PATH

PosixPath('../data/SocialMediaUsersDataset.csv')

In [2]:
df = pd.read_csv(DATA_PATH, dtype={"UserID": "string"})

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 100,000
Columns: ['UserID', 'Name', 'Gender', 'DOB', 'Interests', 'City', 'Country']


,UserID,Name,Gender,DOB,Interests,City,Country
0,00001,Jesse Lawhorn,Female,1958-10-15,"'Movies', 'Fashion', 'Fashion', 'Books'",Sibolga,Indonesia
1,00002,Stacy Payne,Female,2004-07-21,"'Gaming', 'Finance and investments', 'Outdoor ...",Al Abyār,Libya
2,00003,Katrina Nicewander,Female,2000-02-07,"'DIY and crafts', 'Music', 'Science', 'Fashion'",Wādī as Sīr,Jordan
3,00004,Eric Yarbrough,Male,1985-04-14,"'Outdoor activities', 'Cars and automobiles'",Matera,Italy
4,00005,Daniel Adkins,Female,1955-09-18,"'Politics', 'History'",Biruaca,Venezuela


## 1. Missing values and user uniqueness

Check basic completeness and confirm that each user has a unique ID.

In [3]:
missing_values = df.isna().sum().to_frame("missing_count")  # Count of null value per column
missing_values["missing_pct"] = (missing_values["missing_count"] / len(df) * 100).round(4)  # Calculates the percentage of missing values for each column.
missing_values

,missing_count,missing_pct
UserID,0,0.0
Name,0,0.0
Gender,0,0.0
DOB,0,0.0
Interests,0,0.0
City,0,0.0
Country,0,0.0


`UserID` should uniquely identify a user.

In [4]:
user_id_report = pd.DataFrame(  # Create a df to show the uniqueness of users 
    [
        {"metric": "total_rows", "value": len(df)},
        {"metric": "unique_user_ids", "value": df["UserID"].nunique()},
        {"metric": "duplicate_user_id_rows", "value": int(df["UserID"].duplicated().sum())},
    ]
)

user_id_report

,metric,value
0,total_rows,100000
1,unique_user_ids,100000
2,duplicate_user_id_rows,0


## 2. Interest parsing and distribution

Parse the raw interest string into a list, then inspect interest counts, duplicates, and topic frequency.

In [5]:
def parse_interests(raw: str) -> list[str]:
    """
      Convert the raw Interests field into a clean list of interest strings.

      The dataset stores interests as a single string containing quoted,
      comma-separated values, for example:
      "'Gaming', 'Travel', 'Technology'"

      Returns an empty list for missing values and removes surrounding
      whitespace from each parsed interest.
      """
    # Missing interest values cannot contribute to retrieval/ranking features.
    # Return an empty list so downstream validation can count them explicitly.
    if pd.isna(raw):
        return []
    try:
        # Wrap the comma-separated string in brackets so Python can parse it
        # as a list while preserving multi-word labels like "Outdoor activities".
        values = ast.literal_eval("[" + str(raw) + "]")
    except Exception:
        # Fallback for unexpected formatting: split on commas and remove quotes.
        values = [part.strip().strip("\"'") for part in str(raw).split(",")]
    # Keep preprocessing lightweight: trim whitespace, but do not fuzzy-merge
    # categories because this dataset appears to use a controlled vocabulary.
    return [str(value).strip() for value in values if str(value).strip()]


# Parse every user's raw Interests value into a list for validation.
parsed_interests = df["Interests"].apply(parse_interests)
# Count interests per user to catch empty or unusually long interest lists.
interest_lengths = parsed_interests.apply(len)
# Flag users where the same interest appears more than once in the same row.
duplicate_interest_rows = parsed_interests.apply(lambda values: len(values) != len(set(values)))

interest_report = pd.DataFrame(
    [
        {"metric": "rows_with_zero_interests", "value": int((interest_lengths == 0).sum())},
        {"metric": "min_interests_per_user", "value": int(interest_lengths.min())},
        {"metric": "max_interests_per_user", "value": int(interest_lengths.max())},
        {"metric": "rows_with_duplicate_interests", "value": int(duplicate_interest_rows.sum())},
        {"metric": "unique_interests", "value": len(set(interest for values in parsed_interests for interest in values))},
    ]
)

interest_report

,metric,value
0,rows_with_zero_interests,0
1,min_interests_per_user,1
2,max_interests_per_user,5
3,rows_with_duplicate_interests,12897
4,unique_interests,29


### Interest preprocessing note

The dataset uses a small controlled interest vocabulary. Heavy fuzzy matching is not needed.

Use lightweight cleanup only:

- parse interests into a list
- trim whitespace
- deduplicate repeated interests per user
- preserve the original category labels

Avoid semantic merging for now because related labels can still represent different signals.


In [6]:
# Flatten all parsed interest lists and count the most common topics.
top_interests = (
    pd.Series([interest for values in parsed_interests for interest in values])
    .value_counts()
    .head(20)
    .rename_axis("interest")
    .reset_index(name="count")
)

top_interests

,interest,count
0,Fashion,20077
1,Pets,10164
2,Cooking,10163
3,Fitness,10108
4,Movies,10102
5,Gaming,10093
6,Travel,10048
7,Cars and automobiles,10048
8,Outdoor activities,10041
9,Books,10029


In [7]:
# Use the same counts to inspect the least common topics.
# Rare topics can reveal misspellings, fragmented labels, or long-tail categories.
least_interests = (
    pd.Series([interest for values in parsed_interests for interest in values])
    .value_counts()
    .tail(20)
    .sort_values(ascending=True)
    .rename_axis("interest")
    .reset_index(name="count")
)

least_interests


,interest,count
0,Politics,9731
1,Science,9792
2,Beauty,9858
3,DIY and crafts,9872
4,Food and dining,9886
5,Music,9887
6,Parenting and family,9898
7,History,9899
8,Nature,9913
9,Sports,9923


## 3. DOB and gender overview

Check whether DOB can support age features and inspect available gender categories.

In [8]:
dob = pd.to_datetime(df["DOB"], errors="coerce") # converting date as string to pandas datetime for better analysis
as_of = pd.Timestamp.today().normalize()
age = ((as_of - dob).dt.days / 365.25).astype("float") # A normal year has 365 days, but leap years have 366. Since leap years happen roughly every 4 years, the average year length is: (365 + 365 + 365 + 366) / 4 = 365.25

date_gender_report = pd.DataFrame(
    [
        {"metric": "invalid_dob_rows", "value": int(dob.isna().sum())},
        {"metric": "earliest_dob", "value": dob.min().date() if dob.notna().any() else None},
        {"metric": "latest_dob", "value": dob.max().date() if dob.notna().any() else None},
        {"metric": "min_age", "value": int(age.min()) if age.notna().any() else None},
        {"metric": "max_age", "value": int(age.max()) if age.notna().any() else None},
        {"metric": "gender_values", "value": sorted(df["Gender"].dropna().unique().tolist())},
    ]
)

date_gender_report

,metric,value
0,invalid_dob_rows,0
1,earliest_dob,1954-01-01
2,latest_dob,2004-12-30
3,min_age,21
4,max_age,72
5,gender_values,"[Female, Male]"


## 4. Location field overview

Inspect available location fields and confirm whether true geographic distance can be computed from the raw data.

In [9]:
location_report = pd.DataFrame(
    [
        {"metric": "missing_city_rows", "value": int(df["City"].isna().sum())},
        {"metric": "missing_country_rows", "value": int(df["Country"].isna().sum())},
        {"metric": "unique_countries", "value": df["Country"].nunique()},
        {"metric": "unique_cities", "value": df["City"].nunique()},
        {"metric": "unique_city_country_pairs", "value": df[["City", "Country"]].drop_duplicates().shape[0]},
        {"metric": "latitude_column_present", "value": "Latitude" in df.columns},
        {"metric": "longitude_column_present", "value": "Longitude" in df.columns},
    ]
)

location_report

,metric,value
0,missing_city_rows,0
1,missing_country_rows,0
2,unique_countries,244
3,unique_cities,24452
4,unique_city_country_pairs,25076
5,latitude_column_present,False
6,longitude_column_present,False


In [10]:
country_distribution = (
    df["Country"]
    .value_counts()
    .head(15)
    .rename_axis("country")
    .reset_index(name="user_count")
)

country_distribution

,country,user_count
0,United States,12311
1,India,9399
2,China,7381
3,Brazil,4643
4,Russia,4254
5,Germany,4174
6,Japan,3148
7,United Kingdom,3086
8,France,2458
9,Mexico,2445


## 5. Data quality summary

Summarize the checks that affect preprocessing and recommendation design.

In [11]:
summary_checks = [
    ("UserID is unique", df["UserID"].is_unique, int(df["UserID"].duplicated().sum())),
    ("Dataset has no missing values", df.isna().sum().sum() == 0, df.isna().sum().to_dict()),
    ("DOB values are parseable", dob.isna().sum() == 0, int(dob.isna().sum())),
    ("Every row has at least one parsed interest", (interest_lengths > 0).all(), int((interest_lengths == 0).sum())),
    ("City and country are available", df[["City", "Country"]].isna().sum().sum() == 0, df[["City", "Country"]].isna().sum().to_dict()),
    ("Latitude/longitude are not present in the raw dataset", not {"Latitude", "Longitude"}.issubset(df.columns), [col for col in ["Latitude", "Longitude"] if col not in df.columns]),
]

schema_summary = pd.DataFrame(summary_checks, columns=["check", "passed", "details"])
schema_summary

,check,passed,details
0,UserID is unique,True,0
1,Dataset has no missing values,True,"{'UserID': 0, 'Name': 0, 'Gender': 0, 'DOB': 0..."
2,DOB values are parseable,True,0
3,Every row has at least one parsed interest,True,0
4,City and country are available,True,"{'City': 0, 'Country': 0}"
5,Latitude/longitude are not present in the raw ...,True,"[Latitude, Longitude]"


## Key EDA insights

- The dataset has `100,000` rows and `7` raw columns: `UserID`, `Name`, `Gender`, `DOB`, `Interests`, `City`, and `Country`.
- There are no missing values in any raw column, so basic row completeness is strong.
- `UserID` is clean for user-level splitting: `100,000` unique IDs and `0` duplicate user IDs.
- `DOB` values are fully parseable. The observed age range is approximately `21` to `72` based on the dataset.
- `Interests` parses cleanly for every row. Each user has between `1` and `5` interests.
- The interest vocabulary is small: only `29` unique interest topics. This supports the assumption that interests are controlled labels, not messy free-text tags.
- Heavy fuzzy preprocessing is not justified for interests. There are no obvious fragmented rare labels in the top/least frequency tables.
- `12,897` rows contain duplicate interests within the same user row, so per-user interest deduplication is still needed before feature engineering.
- Location fields are available as `City` and `Country`, with `244` countries, `24,452` cities, and `25,076` city-country pairs.
- The dataset also does not include follow/click/like labels, so neural ranking needs a defensible proxy target or synthetic label strategy.
